## Model accuracy
Import libraries

In [1]:
import os
import utils
import numpy as np
import pandas as pd
import tensorflow as tf

from scipy.constants import Avogadro as UNA

Define inputs

In [2]:
model_name = "meno"
# Data indices
indices = {
  "x": [list(range(4)),[4],[5],[6]],
  "y": [list(range(7,11)), [11]]
}
# Paths
path_to_data = "./../00.datagen/data/test/"
path_to_model = "./../03.merged/saving/plasma/SavedModel/"
path_to_saving = "./output/error/"
os.makedirs(path_to_saving, exist_ok=True)

Load data and model

In [ ]:
data = utils.load_test_cases(path_to_data)
model = tf.keras.models.load_model(path_to_model)

Set variables names

In [ ]:
names = np.array(list(data.values())[0].columns)
names = np.concatenate([names[i] for i in indices["y"]]).tolist()
names = [name[2:] for name in names]
names

Compute error

In [5]:
error = {"l2": [], "mape": []}
for idata in data.values():
  t, y_true, y_pred = utils.predict(model, idata.to_numpy(), indices)
  y_true = y_true[1:] / UNA
  y_pred = y_pred[1:] / UNA
  error["l2"].append(utils.error_l2(y_true, y_pred))
  error["mape"].append(utils.error_mape(y_true, y_pred, eps=1e-10))

In [6]:
for (k, err) in error.items():
  kerr = np.vstack(err)
  kerr = np.vstack([np.mean(kerr, axis=0), np.std(kerr, axis=0)])
  kerr = pd.DataFrame(kerr, columns=names, index=["mean", "std"])
  kerr.to_csv(path_to_saving+f"/{model_name}_{k}.csv", float_format="%.2f")  